# 02 — Red Team Batch Evaluation & Guardrail Benchmarks

Runs the 38-probe OWASP LLM Top 10 suite (30 adversarial payloads across `LLM01`, `LLM06`, `LLM08` + 8 benign controls) against both **Unguarded DeepSeek-R1** and the **Dual-Guardrailed Pipeline**, inspecting:
- Chain-of-Thought (`<think>`) hijacking traces
- Reversible PII tokenization (`EMAIL`, `API_KEY`, `SSN`, `IP_ADDRESS`, `IN_AADHAAR`, `IN_PAN`)
- Agentic indirect prompt injection inside `read_inbox()` / `query_db()`
- ASR, FPR, latency overhead, and `reports/asr_comparison.png`

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/daipayanghosal/llm-adversarial-threat-lab.git'
REPO_DIR = Path('/content/llm-adversarial-threat-lab')

if not Path('src/evaluator.py').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

!pip install -q -r requirements.txt

from IPython.display import Image, display
from src.llm_client import OllamaClient
from src.evaluator import run_benchmark

client = OllamaClient()
mode = 'live' if client.is_available() else 'simulate'
print(f'Running evaluation suite in mode={mode!r}')

summary = run_benchmark(mode=mode, print_table=True)

In [ ]:
# Inspect CoT Hijack, Indian Aadhaar/PAN Masking, and Agentic Indirect Injection traces
from src.api_server import EvaluateRequest, evaluate_prompt
from src.evaluator import load_payloads

_, catalog = load_payloads()
for pid, agentic, scen in [
    ('LLM01-02', False, 'clean_inbox'),
    ('LLM02-01', False, 'clean_inbox'),
    ('LLM01-04', True, 'poisoned_invoice_email'),
]:
    item = next(p for p in catalog if p['id'] == pid)
    res = evaluate_prompt(
        EvaluateRequest(
            prompt=item['prompt'],
            payload_id=pid,
            agentic_mode=agentic,
            scenario_id=scen,
        )
    )
    print(f'=== {pid}: {item["title"]} ===')
    print('Unguarded breach:', res['unguarded']['attack_succeeded'])
    print('Guardrail verdict:', res['guardrailed']['verdict'], '| Risk:', res['guardrailed']['pre_filter']['risk_score'])
    print('PII masked:', res['guardrailed']['pii_masking']['counts_by_type'])
    print('Sanitized output:', res['guardrailed']['final_output'][:180], '\n')

In [ ]:
# Render Matplotlib ASR & Latency Overhead Chart
display(Image(filename='reports/asr_comparison.png'))